In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
print("Cargando archivo Excel...")
# Cargar el archivo Excel
df = pd.read_excel("data_train.xlsx")
print("Archivo cargado correctamente.")

Cargando archivo Excel...
Archivo cargado correctamente.


In [3]:
df=df[df['CENIZAS'].notna()] #FILTRAMOS REGISTROS CON CENIZAS NULO
df=df[df['4dw1.ctrl']>50] #FILTRAMOS REGSTROS CON PESO MENOR A 50
df=df[df['GRADO']!='CUADERNOS U/B 56'] #FILTRAMOS EL GRADO CUARDERNOS U/B 56 PUES TIENE CENIZAS ATIPICAS PARA EL PESO
df=df[df['4205JMCM']==1.0] #FILTRAMOS CUANDO LA MAQUINA ESTA PARADA
df=df[df['plc10_0_3']==0] #FILTRAMOS CUANDO LA MAQUINA ESTA REVENTADA


In [4]:
df.drop(columns=['4IS0.MTSREELREAL','plc10_0_3','4205JMCM','GRADO'],index=1,inplace=True)
df.drop(columns=['42nic025','4dw1.ctrl','42FIC103'],inplace=True)
df=df.dropna() #SE ELIMINAN FILAS CON REGISTROS VACIOS

In [5]:
# Confirmar que el archivo fue cargado correctamente
print("Encabezados del archivo cargado:", df.columns.tolist())

# Seleccionar características y variable objetivo
features = ['42nic073', '42fic109', '44fic108', '44dic108', '4.kgtr.agret',
            '42nt122.b', 'cenizas_total', 'ret_1er_paso_mv']
target = 'CENIZAS'

df = df.drop('Timestamp', axis=1)

X = df[features]
y = df[target]

Encabezados del archivo cargado: ['Timestamp', '42nic073', '42fic109', '44fic108', '44dic108', '4.kgtr.agret', '42nt122.b', 'cenizas_total', 'ret_1er_paso_mv', 'CENIZAS']


In [7]:

# Dividir en datos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [8]:

# Definir el modelo Random Forest con los hiperparámetros dados
rf = RandomForestRegressor(n_estimators=300, min_samples_split=5, min_samples_leaf=2, 
                           max_depth=None, random_state=42)

# Entrenar el modelo
print("Entrenando Random Forest con hiperparámetros predefinidos...")
rf.fit(X_train, y_train)

# Evaluar el modelo con los datos de prueba
y_pred = rf.predict(X_test)

# Calcular métricas de error
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"Evaluación del modelo:")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")




Entrenando Random Forest con hiperparámetros predefinidos...
Evaluación del modelo:
MSE: 0.11
RMSE: 0.34
MAE: 0.12


In [ ]:
# Generar gráfico de dispersión
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title(f"Random Forest Ajustado (RMSE: {rmse:.2f}, MAE: {mae:.2f})")
plt.xlabel("Valor Real")
plt.ylabel("Predicción")
plt.show()


# Crear un DataFrame con los resultados usando los índices originales de y_test
resultados_df = pd.DataFrame({
    'Timestamp': df.loc[y_test.index, 'Timestamp'],  # Asegurar que 'Timestamp' esté en el DataFrame original
    'CENIZAS_LAB1': y_test.values,
    'CENIZAS_CALC': y_pred
})

# Guardar los resultados en un archivo Excel
ruta_salida = "D:\Base de datos Cenizas total.xlsx"
#resultados_df.to_excel(ruta_salida, index=False)
print(f"Archivo Excel guardado en: {ruta_salida}")

print("Proceso completado.")

In [9]:
import joblib
joblib.dump(rf,'modelo_random_forest_V3.pkl')

['modelo_random_forest_V3.pkl']

In [16]:
df_test=pd.read_excel('data_val.xlsx')

In [22]:
x_test=df_test.iloc[:,1:-6]

In [24]:
x_test.drop(columns=['42nic025','4dw1.ctrl','42FIC103'],index=1,inplace=True)

In [25]:
x_test

,42nic073,42fic109,44fic108,44dic108,4.kgtr.agret,42nt122.b,cenizas_total,ret_1er_paso_mv
0,3.860676,77.597008,7.847770,35.067802,0.650033,62.029495,93.330147,87.717628
2,3.855635,78.572487,7.861619,35.067802,0.650208,61.753628,93.199646,87.612968
3,3.852972,80.213295,7.848302,35.067802,0.650295,62.728577,93.019829,87.414375
4,3.850309,82.095406,7.850909,35.067802,0.650383,62.633430,93.190346,87.638962
5,3.847646,78.392868,7.834209,35.067802,0.650470,62.830761,93.171501,87.611626
...,...,...,...,...,...,...,...,...
4227,3.682192,107.570335,7.397993,33.713600,0.597535,62.471581,92.774788,86.920265
4228,3.680449,111.454216,7.400463,33.713600,0.597535,62.485958,92.920799,87.083000
4229,3.678706,108.991547,7.403682,33.713600,0.597535,62.449722,92.641899,87.100601
4230,3.676964,106.707634,7.390620,33.713600,0.597535,62.485958,92.913582,87.080620


In [26]:
y_test = rf.predict(x_test)

In [ ]:
list(y_test)

In [27]:
df_new=pd.DataFrame(list(y_test))

In [29]:
df_new.to_excel('prueba_v3.xlsx',index=False)